In [0]:
# ============================================================
# CELL 1 — Bronze Layer Ingestion
# Notebook: 01_bronze_ingestion
# Pipeline: Enterprise Predictive Maintenance & IoT Sensor Optimization
# Data: NASA CMAPSS Turbofan Engine Degradation Dataset
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp

BASE_PATH = "/Volumes/workspace/predictive_maintenance/predictive_maintenance"

# Column names per NASA readme — 26 columns total
# unit_id + time_cycles + 3 operational settings + 21 sensors
col_names = [
    "unit_id", "time_cycles",
    "op_setting_1", "op_setting_2", "op_setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
    "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
    "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
    "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20",
    "sensor_21"
]

def read_cmapss(filename, subset_label, split_label):
    """
    Read a raw CMAPSS txt file from Unity Catalog volume.
    Adds metadata columns: subset (FD001/FD003), split (train/test),
    and ingested_at timestamp for bronze layer lineage.
    Drops trailing null columns created by NASA file whitespace.
    """
    path = f"{BASE_PATH}/{filename}"
    df = (spark.read
          .option("sep", " ")
          .option("header", "false")
          .option("inferSchema", "true")
          .csv(path)
          .toDF(*col_names, "null_1", "null_2")  # 2 trailing null cols
          .drop("null_1", "null_2")
          .withColumn("subset", lit(subset_label))
          .withColumn("split", lit(split_label))
          .withColumn("ingested_at", current_timestamp())
    )
    return df

def read_rul(filename, subset_label):
    """Read RUL ground truth file — single column with engine index as row number."""
    path = f"{BASE_PATH}/{filename}"
    df = (spark.read
          .option("sep", " ")
          .option("header", "false")
          .option("inferSchema", "true")
          .csv(path)
          .toDF("rul_true", "null_1")
          .drop("null_1")
          .withColumn("subset", lit(subset_label))
          .withColumn("ingested_at", current_timestamp())
    )
    return df

# --- Ingest all 6 files ---
train_fd001 = read_cmapss("train_FD001.txt", "FD001", "train")
train_fd003 = read_cmapss("train_FD003.txt", "FD003", "train")
test_fd001  = read_cmapss("test_FD001.txt",  "FD001", "test")
test_fd003  = read_cmapss("test_FD003.txt",  "FD003", "test")
rul_fd001   = read_rul("RUL_FD001.txt", "FD001")
rul_fd003   = read_rul("RUL_FD003.txt", "FD003")

# --- Sanity checks ---
print("=== BRONZE LAYER INGESTION SUMMARY ===")
print(f"train_FD001 rows: {train_fd001.count():,}  | cols: {len(train_fd001.columns)}")
print(f"train_FD003 rows: {train_fd003.count():,}  | cols: {len(train_fd003.columns)}")
print(f"test_FD001  rows: {test_fd001.count():,}  | cols: {len(test_fd001.columns)}")
print(f"test_FD003  rows: {test_fd003.count():,}  | cols: {len(test_fd003.columns)}")
print(f"rul_FD001   rows: {rul_fd001.count():,}   | cols: {len(rul_fd001.columns)}")
print(f"rul_FD003   rows: {rul_fd003.count():,}   | cols: {len(rul_fd003.columns)}")
print("\n=== SCHEMA (train_FD001) ===")
train_fd001.printSchema()
print("\n=== SAMPLE ROWS (train_FD001) ===")
train_fd001.show(5, truncate=False)

=== BRONZE LAYER INGESTION SUMMARY ===
train_FD001 rows: 20,631  | cols: 29
train_FD003 rows: 24,720  | cols: 29
test_FD001  rows: 13,096  | cols: 29
test_FD003  rows: 16,596  | cols: 29
rul_FD001   rows: 100   | cols: 3
rul_FD003   rows: 100   | cols: 3

=== SCHEMA (train_FD001) ===
root
 |-- unit_id: integer (nullable = true)
 |-- time_cycles: integer (nullable = true)
 |-- op_setting_1: double (nullable = true)
 |-- op_setting_2: double (nullable = true)
 |-- op_setting_3: double (nullable = true)
 |-- sensor_1: double (nullable = true)
 |-- sensor_2: double (nullable = true)
 |-- sensor_3: double (nullable = true)
 |-- sensor_4: double (nullable = true)
 |-- sensor_5: double (nullable = true)
 |-- sensor_6: double (nullable = true)
 |-- sensor_7: double (nullable = true)
 |-- sensor_8: double (nullable = true)
 |-- sensor_9: double (nullable = true)
 |-- sensor_10: double (nullable = true)
 |-- sensor_11: double (nullable = true)
 |-- sensor_12: double (nullable = true)
 |-- sensor

In [0]:
# ============================================================
# CELL 2 — Write Bronze Delta Tables to Unity Catalog
# ============================================================

# Combine train + test for each subset into single bronze tables
from pyspark.sql.functions import col

# Union train and test for each subset
bronze_fd001 = train_fd001.union(test_fd001)
bronze_fd003 = train_fd003.union(test_fd003)

# Write as Delta tables in the predictive_maintenance schema
(bronze_fd001.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.predictive_maintenance.bronze_fd001"))

(bronze_fd003.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.predictive_maintenance.bronze_fd003"))

(rul_fd001.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.predictive_maintenance.bronze_rul_fd001"))

(rul_fd003.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.predictive_maintenance.bronze_rul_fd003"))

print("✅ Bronze Delta tables written successfully:")
print("   - workspace.predictive_maintenance.bronze_fd001")
print("   - workspace.predictive_maintenance.bronze_fd003")
print("   - workspace.predictive_maintenance.bronze_rul_fd001")
print("   - workspace.predictive_maintenance.bronze_rul_fd003")

✅ Bronze Delta tables written successfully:
   - workspace.predictive_maintenance.bronze_fd001
   - workspace.predictive_maintenance.bronze_fd003
   - workspace.predictive_maintenance.bronze_rul_fd001
   - workspace.predictive_maintenance.bronze_rul_fd003
